# Kaggle evaluation run — finalproject_cpe027

Runs the Tier 1 (ACL6060 + seeded DEMAND noise) and Tier 2 (unmodified M3AV) ASR evaluation
on Kaggle (GPU) using preprocessed local data. No preprocessing is redone here.

- Tiers are executed and reported separately; they are never pooled.
- Checkpoints are retained per tier; this notebook starts fresh (`--no-resume`).
- Whisper: `base`, `cuda`, `float16`, `en` (Tier 1). DSP/decoding settings come from the runner.
- Data mounts are discovered recursively (nested under /kaggle/input/datasets/<user>/...).


In [63]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

# ---- resolve project code (attached dataset first, git fallback) ---------
REPO_DIR = None
for p in Path("/kaggle/input").rglob("backend"):
    if (p / "app").is_dir():
        REPO_DIR = p.parent
        break
REPO_DIR = REPO_DIR or Path("/kaggle/working/finalproject_cpe027")
BACKEND_DIR = REPO_DIR / "backend"

REPO_GIT_URL = os.environ.get(
    "REPO_GIT_URL", "https://github.com/leon-pscl/finalproject_cpe027.git"
).strip()
if not REPO_DIR.is_dir():
    print("Cloning project repository ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_GIT_URL, str(REPO_DIR)], check=True
    )
if not (BACKEND_DIR / "app").is_dir():
    raise FileNotFoundError(
        "Backend directory not found. Attach a dataset containing backend/app "
        "or make the repository public and ensure git can clone it."
    )

# ---- resolve data mounts recursively (nested under datasets/<user>) ----
INPUT = Path("/kaggle/input")
SEARCH_ROOTS = [INPUT, *[p for p in INPUT.glob("datasets/*") if p.is_dir()],
                 *[p for p in INPUT.glob("datasets/*/*") if p.is_dir()]]

acl_hits = []
for p in SEARCH_ROOTS:
    cand = (p / "dev" / "full_wavs")
    if cand.is_dir():
        acl_hits.append(p)
acl_hits = list(dict.fromkeys(acl_hits))
if not acl_hits:
    listing = sorted(str(p.name) for p in INPUT.iterdir()) if INPUT.is_dir() else []
    raise FileNotFoundError(
        "ACL6060 data not found. Scanned roots: " + ", ".join(str(p) for p in SEARCH_ROOTS)
        + ". Top-level entries: " + ", ".join(listing)
    )

demand_hits = []
for p in SEARCH_ROOTS:
    for cand in (p, p / "DEMAND"):
        if (cand / "DKITCHEN_16k").is_dir() and next(cand.rglob("*.wav"), None) is not None:
            demand_hits.append(cand)
demand_hits = list(dict.fromkeys(demand_hits))
if not demand_hits:
    listing = sorted(str(p.name) for p in INPUT.iterdir()) if INPUT.is_dir() else []
    raise FileNotFoundError(
        "DEMAND data not found. Scanned roots: " + ", ".join(str(p) for p in SEARCH_ROOTS)
        + ". Top-level entries: " + ", ".join(listing)
    )

m3av_raw = sorted({str(p / "M3AV" / "extracted")
                for p in SEARCH_ROOTS
                if (p / "M3AV" / "extracted").is_dir()})
m3av_zips = sorted({z for p in SEARCH_ROOTS for z in p.rglob("*.zip") if "m3av" in z.name.lower()})

ACL_ROOT = acl_hits[0]
DEMAND_ROOT = demand_hits[0]

os.environ["REPO_DIR"] = str(REPO_DIR)
os.environ["BACKEND_DIR"] = str(BACKEND_DIR)
os.environ["ACL_ROOT"] = str(ACL_ROOT)
os.environ["DEMAND_ROOT"] = str(DEMAND_ROOT)
M3AV_ZIPS = [str(z) for z in m3av_zips]
os.environ["M3AV_ZIPS"] = chr(10).join(M3AV_ZIPS)
os.environ["M3AV_CLIP_ROOT"] = "\n".join(m3av_raw)

print("ACL root:", ACL_ROOT)
print("DEMAND root:", DEMAND_ROOT)
print("M3AV raw roots:", m3av_raw)
print("M3AV zips (lazy-extract):", [z.name for z in m3av_zips], flush=True)
!nvidia-smi


ACL root: /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
DEMAND root: /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand/DEMAND
M3AV raw roots: ['/kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-0/M3AV/extracted', '/kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-1/M3AV/extracted']
M3AV zips (lazy-extract): []
Fri Nov  6 22:28:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|==================

In [64]:
!df -h

Filesystem                         Size  Used Avail Use% Mounted on
overlay                            8.0T  7.1T  967G  89% /
tmpfs                               64M     0   64M   0% /dev
shm                                 14G  4.0K   14G   1% /dev/shm
/dev/loop2                          20G  659M   19G   4% /kaggle/lib
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-0
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-m3av-split-1
kkb-production-mirror-us-central1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
/dev/sda1                          122G  7.0G  115G   6% /opt/bin
/dev/mapper/snap                   8.0T  7.1T  967G  89% /etc/hosts
tmpfs                        

In [65]:
from pathlib import Path

print("Input directory tree (up to depth 3):")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if not p.is_dir():
        continue
    rel = p.relative_to("/kaggle/input")
    if len(rel.parts) <= 3:
        print("  " * (len(rel.parts) - 1) + rel.name + "/")


Input directory tree (up to depth 3):
datasets/
  leonpascual1/
    finalproject-cpe027-eval-acl6060/
    finalproject-cpe027-eval-demand/
    finalproject-cpe027-eval-m3av-split-0/
    finalproject-cpe027-eval-m3av-split-1/


In [66]:
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12==9.*

In [67]:
!pip install -q faster-whisper av scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic

In [70]:
os.environ["ENV"] = "production"
os.environ["PROCESSING_MODE"] = "real"
# arm_a SNR fix: forced signal_snr_improvement_db=0.0 in eval_runner (see docs)
import os

os.environ["WHISPER_MODEL"] = "base"
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
os.environ["WHISPER_THREADS"] = "4"
os.environ["WHISPER_LANGUAGE"] = "en"
os.environ["MODEL_CACHE_DIR"] = "/kaggle/working/models"
os.environ["EVAL_DATA_ROOT"] = os.environ.get("ACL_ROOT", "/kaggle/input/acl6060")
# arm_a SNR fix: forced signal_snr_improvement_db=0.0 in eval_runner (see docs)
# Attach M3AV split zip paths (Tier 2)
os.environ["M3AV_ZIPS"] = "E:\\Datasets\\kaggle_datasets\\m3av_0\\m3av_0.zip\nE:\\Datasets\\kaggle_datasets\\m3av_1\\m3av_1.zip\n"

# ---- Verify CUDA / CTranslate2 capability (GPU fallback check) ----
try:
    import ctranslate2
    print("ctranslate2:", ctranslate2.__version__)
    print("CUDA compute types:", ctranslate2.get_supported_compute_types("cuda"))
    print("CPU compute types:", ctranslate2.get_supported_compute_types("cpu"))
except Exception as exc:
    print("CTranslate2 CUDA check failed:", type(exc).__name__, exc)
try:
    import torch
    print("torch CUDA available:", torch.cuda.is_available(), "count:", torch.cuda.device_count())
except Exception as exc:
    print("torch CUDA check failed:", type(exc).__name__, exc)

ctranslate2: 4.8.5
CUDA compute types: {'float16', 'int8_float16', 'int8_float32', 'int8', 'float32'}
CPU compute types: {'int8', 'float32', 'int16', 'int8_float32'}
torch CUDA available: True count: 2


In [71]:
import subprocess
import sys
from importlib.metadata import version


def av_ok() -> bool:
    """Probe in a fresh interpreter so we see the currently installed av."""
    probe = subprocess.run(
        [sys.executable, "-c",
         "import av; av.open('/nonexistent.wav', metadata_errors='ignore')"],
        capture_output=True, text=True,
    )
    last = (probe.stderr.strip().splitlines() or [""])[-1]
    return "unexpected keyword argument 'metadata_errors'" not in last


if not av_ok():
    print(f"av {version('av')} is incompatible with faster-whisper "
          f"{version('faster-whisper')}; trying older prebuilt wheels ...")
    fixed = False
    # Newest prebuilt wheel below each ceiling; never compile from source.
    for ceiling in ["19", "17", "16", "15", "14", "13", "12"]:
        spec = f"av<{ceiling}"
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--only-binary=:all:", spec],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print(f"  {spec}: no usable wheel")
            continue
        ok = av_ok()
        print(f"  {spec}: installed av {version('av')} -> {'OK' if ok else 'still incompatible'}")
        if ok:
            fixed = True
            break
    if not fixed:
        raise RuntimeError(
            "No prebuilt av wheel works with this faster-whisper. Options: "
            "(1) `!python -m pip install -q 'faster-whisper<1.1'` and re-run this cell, or "
            "(2) paste this output so we can pick another pin."
        )
print("av/faster-whisper compatibility: OK\n")

for pkg in ["ctranslate2", "faster-whisper", "av", "scipy", "numpy", "pandas",
            "rapidfuzz", "matplotlib", "pydantic", "pydantic_settings"]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception:
        print(f"{pkg}: unknown")

try:
    import torch
    print("torch:", torch.__version__)
except Exception:
    print("torch: not installed")

av/faster-whisper compatibility: OK

ctranslate2: 4.8.5
faster-whisper: 1.2.1
av: 18.1.0
scipy: 1.16.3
numpy: 2.1.3
pandas: 2.3.3
rapidfuzz: 3.14.6
matplotlib: 3.10.0
pydantic: 2.14.1
pydantic_settings: 2.16.2
torch: 2.11.0+cu130


In [72]:
import os, shutil, glob

# See what's cached and how big model.bin is
for p in glob.glob(os.path.expanduser("~/.cache/huggingface/hub/models--*whisper*")):
    print(p)
    os.system(f"du -sh {p}/snapshots/*/model.bin")

In [74]:
!find / -name "model.bin" -size -1M 2>/dev/null
!find / -name "model.bin" 2>/dev/null | xargs ls -lh

lrwxrwxrwx 1 root root 76 Nov  6 19:45 /kaggle/working/models/models--Systran--faster-whisper-base/snapshots/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66/model.bin -> ../../blobs/d01c3014881c9c6f3133c182f3d2887eb6ca1c789a7538c5c007196857a0a6a9


In [75]:
!grep -n "whisper_model\|download_root\|cache" /kaggle/working/finalproject_cpe027/backend/app/asr.py /kaggle/working/finalproject_cpe027/backend/app/config.py

/kaggle/working/finalproject_cpe027/backend/app/asr.py:77:                "Loading faster-whisper model %s with %s thread(s)", s.whisper_model, threads
/kaggle/working/finalproject_cpe027/backend/app/asr.py:83:                s.whisper_model,
/kaggle/working/finalproject_cpe027/backend/app/asr.py:86:                download_root=s.model_cache_dir,
/kaggle/working/finalproject_cpe027/backend/app/config.py:14:from functools import lru_cache
/kaggle/working/finalproject_cpe027/backend/app/config.py:75:    whisper_model: str = "base"
/kaggle/working/finalproject_cpe027/backend/app/config.py:81:    model_cache_dir: str = "/models"
/kaggle/working/finalproject_cpe027/backend/app/config.py:119:@lru_cache(maxsize=1)
/kaggle/working/finalproject_cpe027/backend/app/config.py:124:def clear_settings_cache() -> None:
/kaggle/working/finalproject_cpe027/backend/app/config.py:126:    get_settings.cache_clear()


In [77]:
!df -h /kaggle/working /tmp
!find / -name "model.bin" 2>/dev/null | xargs -r ls -lh

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  659M   19G   4% /kaggle/working
overlay         8.0T  7.1T  967G  89% /
lrwxrwxrwx 1 root root 76 Nov  6 19:45 /kaggle/working/models/models--Systran--faster-whisper-base/snapshots/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66/model.bin -> ../../blobs/d01c3014881c9c6f3133c182f3d2887eb6ca1c789a7538c5c007196857a0a6a9


In [78]:
!ls -lhL /kaggle/working/models/models--Systran--faster-whisper-base/snapshots/*/
!ls -lh /kaggle/working/models/models--Systran--faster-whisper-base/blobs/

total 142M
-rw-r--r-- 1 root root 2.3K Nov  6 19:45 config.json
-r--r--r-- 1 root root 139M Nov  6 19:45 model.bin
-rw-r--r-- 1 root root 2.0K Nov  6 19:45 README.md
-rw-r--r-- 1 root root 2.2M Nov  6 19:45 tokenizer.json
-rw-r--r-- 1 root root 450K Nov  6 19:45 vocabulary.txt
total 2.6M
-rw-r--r-- 1 root root 2.2M Nov  6 19:45 7818adb6de9fa3064d3ff81226fdd675be1f6344
-rw-r--r-- 1 root root 2.3K Nov  6 19:45 867cf1a0fece1394e01d55e287ba2f09a577c046
-rw-r--r-- 1 root root 1.5K Nov  6 19:45 c7d9f3332a950355d5a77d85000f05e6f45435ea
-rw-r--r-- 1 root root 450K Nov  6 19:45 c9074644d9d1205686f16d411564729461324b75
-rw-r--r-- 1 root root 2.0K Nov  6 19:45 cddfec65ffb3e3df45d2234f09f3d261bf9a9c0a
lrwxrwxrwx 1 root root   79 Nov  6 19:45 d01c3014881c9c6f3133c182f3d2887eb6ca1c789a7538c5c007196857a0a6a9 -> ../../blobs/0c/0c4d8d814973160f53608ff0aa1618a90abbce9ed632bbc54930475ba82ce973


In [79]:
import shutil
shutil.rmtree("/kaggle/working/models/models--Systran--faster-whisper-base", ignore_errors=True)

from huggingface_hub import snapshot_download
path = snapshot_download("Systran/faster-whisper-base", cache_dir="/kaggle/working/models", force_download=True)
print(path)
!ls -lhL {path}

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/kaggle/working/models/models--Systran--faster-whisper-base/snapshots/ebe41f70d5b6dfa9166e2c581c45c9c0cfc57b66
total 142M
-rw-r--r-- 1 root root 2.3K Nov  6 22:29 config.json
-r--r--r-- 1 root root 139M Nov  6 22:29 model.bin
-rw-r--r-- 1 root root 2.0K Nov  6 22:29 README.md
-rw-r--r-- 1 root root 2.2M Nov  6 22:29 tokenizer.json
-rw-r--r-- 1 root root 450K Nov  6 22:29 vocabulary.txt


In [80]:
!MODEL_CACHE_DIR=/kaggle/working/models python -m backend.app.eval_runner <your usual args>

/bin/bash: -c: line 1: syntax error near unexpected token `newline'
/bin/bash: -c: line 1: `MODEL_CACHE_DIR=/kaggle/working/models python -m backend.app.eval_runner <your usual args>'


## Tier 1 smoke test


In [81]:
!pip install -q faster-whisper "av==18.1.0" scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic
from importlib.metadata import version
print("av:", version("av"), "| faster-whisper:", version("faster-whisper"))

av: 18.1.0 | faster-whisper: 1.2.1


In [82]:
import glob
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=True)

# Locate the installed libs and prepend them to LD_LIBRARY_PATH.
# Shell commands (!python ...) inherit this, so the runner subprocess sees it.
site = next(p for p in sys.path if p.endswith("dist-packages") and os.path.isdir(os.path.join(p, "nvidia")))
lib_dirs = glob.glob(os.path.join(site, "nvidia", "cublas", "lib")) + \
           glob.glob(os.path.join(site, "nvidia", "cudnn", "lib"))
print("lib dirs:", lib_dirs)

os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

print(glob.glob(os.path.join(site, "nvidia", "cublas", "lib", "libcublas.so.12*")))

lib dirs: ['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib', '/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib']
['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12']


In [83]:
import os
from pathlib import Path

ACL_ROOT = os.environ.get("ACL_ROOT")
DEMAND_ROOT = os.environ.get("DEMAND_ROOT")
BACKEND_DIR = os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend")

if not (ACL_ROOT and DEMAND_ROOT and Path(ACL_ROOT).is_dir() and Path(DEMAND_ROOT).is_dir()):
    raise RuntimeError("Attach ACL6060 and DEMAND datasets before the smoke test.")

os.chdir(BACKEND_DIR)
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
!python -m app.eval_runner --tier 1 --limit 1 --split eval --conditions arm_a,hp --snrs 0 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root "$ACL_ROOT" --demand-root "$DEMAND_ROOT" --out /kaggle/working/eval_smoke_tier1


Loading ASR model...
Tier 1: 5 talks, SNRs=[0.0]
  2022.acl-long.111 | demand:DKITCHEN | SNR=0.0 | already complete; skipping
  2022.acl-long.111 | demand:DLIVING | SNR=0.0 | already complete; skipping
  2022.acl-long.111 | demand:DWASHING | SNR=0.0 | already complete; skipping
{
  "records": 6,
  "summaries": 1,
  "out": "/kaggle/working/eval_smoke_tier1"
}


In [48]:
import json
from pathlib import Path

t1 = json.loads(Path("/kaggle/working/eval_smoke_tier1/tier1_acl_results.json").read_text())
print("SMOKE tier1 records:", len(t1))
r = t1[0]
print(f"SMOKE WER: {r["wer"]:.2%} | RTF: {r["rtf"]:.3f} | seconds: {r["seconds"]:.1f}")


SMOKE tier1 records: 6
SMOKE WER: 26.69% | RTF: 0.064 | seconds: 45.7


## Full run  Tier 1 (ACL6060 + seeded DEMAND noise)

All six conditions, SNRs `none,0,5,10,20`, DEMAND envs `DKITCHEN,DLIVING,DWASHING`. Fresh run (`--no-resume`).

In [22]:
!grep -n -i -E "resume|checkpoint" /kaggle/working/finalproject_cpe027/backend/app/eval_runner.py | head -40

113:CHECKPOINT_SCHEMA_VERSION = 2
118:class CheckpointIncompatibleError(RuntimeError):
119:    """Raised when a checkpoint cannot be safely resumed for the current run."""
200:def _load_checkpoint(path: Path, expected_manifest: dict[str, Any] | None = None) -> dict[str, Any]:
202:        return {"schema_version": CHECKPOINT_SCHEMA_VERSION, "records": [], "run_manifest": expected_manifest}
207:        raise CheckpointIncompatibleError(f"incompatible checkpoint (unreadable): {exc}") from exc
209:        raise CheckpointIncompatibleError("incompatible checkpoint (missing records list)")
211:    if schema_version != CHECKPOINT_SCHEMA_VERSION:
212:        raise CheckpointIncompatibleError(
213:            f"incompatible checkpoint schema {schema_version}; "
214:            "start a new run with --no-resume or remove the checkpoint"
217:        raise CheckpointIncompatibleError("incompatible checkpoint manifest/fingerprint")
219:        # Existing checkpoint must be validated by its own mani

In [ ]:
import os
from pathlib import Path

ACL_ROOT = os.environ.get("ACL_ROOT")
DEMAND_ROOT = os.environ.get("DEMAND_ROOT")
if not ACL_ROOT or not Path(ACL_ROOT).is_dir():
    raise RuntimeError("Attach ACL6060 data before Tier 1")
if not DEMAND_ROOT or not Path(DEMAND_ROOT).is_dir():
    raise RuntimeError("Attach DEMAND data before Tier 1")
os.chdir(os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend"))
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
!python -m app.eval_runner --tier 1 --split eval --conditions arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa --snrs none,0,5,10,20 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root "$ACL_ROOT" --demand-root "$DEMAND_ROOT" --out /kaggle/working/batches/tier1_eval


## Tier 2 smoke test


In [ ]:
import os, shutil, zipfile
from pathlib import Path

BACKEND_DIR = os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend")
M3AV_ROOT = os.environ.get("M3AV_CLIP_ROOT", "").splitlines()[0] if os.environ.get("M3AV_CLIP_ROOT", "").splitlines() else ""

if not (M3AV_ROOT and Path(M3AV_ROOT).is_dir()):
    zips = [z for z in os.environ.get("M3AV_ZIPS", "").splitlines() if z.strip()]
    if not zips:
        raise RuntimeError("Attach at least one M3AV split dataset before the smoke test.")
    stamp = Path("/kaggle/working/mounts/_smoke_m3av")
    if not (stamp / "M3AV" / "extracted").is_dir():
        if stamp.exists(): shutil.rmtree(stamp)
        stamp.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(stamp)
    M3AV_ROOT = str(stamp / "M3AV" / "extracted")

os.chdir(BACKEND_DIR)
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
!python -m app.eval_runner --tier 2 --conditions arm_a,hp --snrs 0 --m3av-root "$M3AV_ROOT" --out /kaggle/working/eval_smoke_tier2 --limit 1 --no-resume


In [33]:
import json
from pathlib import Path

t2 = json.loads(Path("/kaggle/working/eval_smoke_tier2/tier2_m3av_results.json").read_text())
print("SMOKE tier2 records:", len(t2))
r2 = t2[0]
print(f"SMOKE tier2 WER: {r2["wer"]:.2%} | RTF: {r2["rtf"]:.3f} | NSHR: {r2["nshr"]}")


SMOKE tier2 records: 2
SMOKE tier2 WER: 27.55% | RTF: 0.072 | NSHR: 0.0


## Full run  Tier 2 (M3AV), batched per duration bucket

In [84]:
import ctypes, glob, os, site

# Locate the pip-installed NVIDIA libs and load them into the process
# (setting LD_LIBRARY_PATH after Python starts does NOT work, so preload instead)
sp = site.getsitepackages()[0]
for pattern in [
    "nvidia/cublas/lib/libcublas.so.12*",
    "nvidia/cublas/lib/libcublasLt.so.12*",
    "nvidia/cudnn/lib/libcudnn*.so.9*",
]:
    for lib in sorted(glob.glob(os.path.join(sp, pattern))):
        try:
            ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
        except OSError as e:
            print("skip", lib, e)

# sanity check
ctypes.CDLL("libcublas.so.12")
print("cuBLAS 12 loaded")

cuBLAS 12 loaded


In [85]:
import os
import shutil
import sys
import zipfile
from pathlib import Path
from types import SimpleNamespace

BACKEND_DIR = os.environ.get("BACKEND_DIR", os.path.join("/kaggle/working", "finalproject_cpe027", "backend"))
os.chdir(BACKEND_DIR)
sys.path.insert(0, BACKEND_DIR)
from app import eval_data, eval_runner

INPUT = Path("/kaggle/input")
raw_roots = sorted({
    str(p.parent.parent) for p in INPUT.rglob("extracted")
    if p.is_dir() and p.parent.name == "M3AV"
})
env_roots = [r for r in os.environ.get("M3AV_CLIP_ROOT", "").split(chr(10)) if r.strip()]
m3av_zips = [z for z in os.environ.get("M3AV_ZIPS", "").split(chr(10)) if z.strip()]

if raw_roots:
    inputs = raw_roots
elif env_roots:
    inputs = env_roots
elif m3av_zips:
    inputs = m3av_zips
else:
    raise RuntimeError("No M3AV splits are attached; Tier 2 cannot run.")

CONDITIONS = "arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa"
BUCKETS = ["<5m", "5-10m", "10-20m"]
MOUNTS = Path("/kaggle/working/mounts")

def extracted_root(zip_path):
    dest = MOUNTS / ("m3av_" + Path(zip_path).stem)
    marker = dest / "M3AV" / "extracted"
    if not marker.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting", Path(zip_path).name, flush=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
    return dest / "M3AV" / "extracted"

for src in inputs:
    if str(src).endswith(".zip"):
        clip_root = extracted_root(src)
        zip_source = src
    else:
        clip_root = Path(src)
        zip_source = None
        if not clip_root.is_dir():
            print("Skip missing root:", clip_root)
            continue

    for bucket in BUCKETS:
        selected = [c for c in eval_data.load_m3av_clips(str(clip_root))
                    if eval_data.duration_bucket(c.duration_seconds) == bucket]
        if not selected:
            continue
        out_name = Path(src).stem if str(src).endswith(".zip") else clip_root.name
        out_dir = "/kaggle/working/batches/tier2_" + \
            bucket.replace("<", "lt_").replace(">", "gt_").replace("+", "p") + "_" + out_name
        orig_loader = eval_runner.load_m3av_clips
        eval_runner.load_m3av_clips = lambda data_root=None, _sel=selected: _sel
        args = SimpleNamespace(
            tier="2",
            conditions=CONDITIONS,
            m3av_root=str(clip_root),
            data_root=None,
            out=out_dir,
            limit=None,
            resume=True,
        )
        try:
            records, _summary = eval_runner.run_tier2(args)
        finally:
            eval_runner.load_m3av_clips = orig_loader
        print(
            "tier2 %s bucket %s: %d records -> %s"
            % (clip_root.name, bucket, len(records), out_dir)
        )

    if zip_source is not None:
        shutil.rmtree(clip_root.parent.parent, ignore_errors=True)
        print("Cleaned", clip_root.parent.parent, flush=True)


Loading ASR model...
Tier 2: 63 clips
  CHI-004BD | already complete; skipping
  CHI-02908 | already complete; skipping
  CHI-02EA8 | already complete; skipping
  CHI-03E0E | already complete; skipping
  CHI-04664 | already complete; skipping
  CHI-05790 | already complete; skipping
  CHI-05B75 | already complete; skipping
  CHI-06BAD | already complete; skipping
  CHI-06F5A | already complete; skipping
  CHI-0755F | already complete; skipping
  CHI-07C31 | already complete; skipping
  CHI-07FCB | already complete; skipping
  CHI-08671 | already complete; skipping
  CHI-08FC4 | already complete; skipping
  CHI-09BEE | already complete; skipping
  CHI-0A6E7 | already complete; skipping
  CHI-0A9AD | already complete; skipping
  CHI-0B493 | already complete; skipping
  CHI-0E981 | already complete; skipping
  CHI-14B6F | already complete; skipping
  CHI-15C84 | already complete; skipping
  CHI-17856 | already complete; skipping
  CHI-1809F | already complete; skipping
  CHI-1902D | alrea

## Merge batch outputs into final results

In [90]:
import os
from pathlib import Path

REPO_DIR = os.environ.get("REPO_DIR", "/kaggle/working/finalproject_cpe027")
MERGE_SCRIPT = Path(REPO_DIR) / "kaggle" / "merge_eval_results.py"
if not MERGE_SCRIPT.is_file():
    MERGE_SCRIPT = Path(REPO_DIR) / "merge_eval_results.py"
os.chdir(REPO_DIR)

!python {MERGE_SCRIPT} --batch-dir /kaggle/working/batches --out /kaggle/working/final


{
  "tier1_records": 0,
  "tier2_records": 1800,
  "tier1_batches": 0,
  "tier2_batches": 4,
  "out": "/kaggle/working/final"
}


In [94]:
from pathlib import Path

import json

names = [
    "tier1_acl_results.json",
    "tier1_acl_results.csv",
    "tier1_acl_summary.json",
    "tier2_m3av_results.json",
    "tier2_m3av_results.csv",
    "tier2_m3av_summary.json",
]
final = Path("/kaggle/working/final")
for name in names:
    p = final / name
    print(name, "OK" if p.is_file() else f"MISSING: {p}")

t1_path = final / "tier1_acl_results.json"
t2_path = final / "tier2_m3av_results.json"
if not t2_path.is_file():
    raise RuntimeError("Final tier JSON files not found; merge step may have failed")
t2 = json.loads(t2_path.read_text(encoding="utf-8"))
print("Tier 1 records:", len(t1))
print("Tier 2 records:", len(t2))
if len(t2) == 0:
    raise RuntimeError("Final Tier 2 JSON has 0 records")


tier1_acl_results.json OK
tier1_acl_results.csv OK
tier1_acl_summary.json OK
tier2_m3av_results.json OK
tier2_m3av_results.csv OK
tier2_m3av_summary.json OK
Tier 1 records: 0
Tier 2 records: 1800
